# Study on Parallel pattern of agentic programming

## Prepare

In [ ]:
import langchain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import Runnable, RunnableParallel, RunnablePassthrough

print(f'Import LangChain V{langchain.__version__}')

Import LangChain V1.2.7


In [ ]:
with open('api-key/deepseek.txt', 'r') as f:
    llm = ChatOpenAI(
        base_url="https://api.deepseek.com",
        api_key=f.read().strip(),
        model="deepseek-v4-flash",
        temperature=0,
    )

# print(
#     StrOutputParser().invoke(
#         llm.invoke([("human", "Introduce yourself briefly")])
#     )
# )

Hi there! I'm DeepSeek, an AI assistant created by DeepSeek (深度求索). I'm here to help you with a wide range of tasks—whether it's answering questions, writing, brainstorming, analyzing documents, or just having a thoughtful conversation. I'm free, support a 1M token context window (think: processing entire book-length texts), and can handle file uploads, links, and voice input on the app. 

Ask me anything—I'm ready to dive in! 😊


## Create 3 parallel chains

In [ ]:
# chain to summarize topic
sum_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Please summarize the following topic briefly"),
        ("user", "{topic}")
    ]) | llm | StrOutputParser()
)

# chain to create questions dut to given topic
que_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Generate three related questions due to given topic"),
        ("user", "{topic}")
    ]) | llm | StrOutputParser()
)

# chain to insight key words due to given topic
key_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Discover 5~10 key terms due to given topic"),
        ("user", "{topic}")
    ]) | llm | StrOutputParser()
)

In [5]:
# generate parallel part
parallel_chain = RunnableParallel({
    "summarize": sum_chain,
    "questions": que_chain,
    "key_terms": key_chain,
    "topic": RunnablePassthrough(), # pass original topic
})

## Create synthesis chain and total chain

In [6]:
syn_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", """Synthesis following info and original topic to generate final response, use Chinese:
         - summazie: {summarize}
         - related questions: {questions}
         - key terms: {key_terms}"""),
        ("user", "origin topic: {topic}")
    ]) | llm | StrOutputParser()
)

In [7]:
complete_chain = parallel_chain | syn_chain

## Test

In [8]:
async def run_async_chain(chain: Runnable, topic: str) -> None:
    print(f"User input: {topic}")
    try:
        response = await chain.ainvoke({"topic": topic})
        print(f"Response: {response}")
    except Exception as e:
        print(f"Failed to run chain: {e}")
    finally:
        print()

In [9]:
test_bench = [
    "介绍一下太空探索的历史，包括前苏联、美国、欧洲、日本和中国的探索活动",
    "思考人口出生率与城市化率的关系",
    "量子计算的愿景目标以及当前的困境",
]

In [11]:
for topic in test_bench:
    await run_async_chain(complete_chain, topic)

User input: 介绍一下太空探索的历史，包括前苏联、美国、欧洲、日本和中国的探索活动
Response: 好的，根据您提供的详尽资料和核心主题，以下是用中文整合生成的最终回答，系统梳理了前苏联、美国、欧洲、日本和中国在太空探索历史中的关键活动与里程碑。

---

### 太空探索历史全景：从冷战竞赛到合作共赢

太空探索的历史是一部人类突破地球束缚、迈入宇宙的壮丽史诗。其起点源于冷战时期两大超级大国——前苏联与美国的激烈竞赛，并逐步演变为包括欧洲、日本、中国等在内的多方力量共同参与的全球性事业。探索的焦点也从早期的“谁先做到”，转向了“如何做得更好”以及“如何携手探索”。

#### 一、 前苏联/俄罗斯：开创与奠基（太空竞赛的先行者）

前苏联在太空时代初期扮演了无可替代的开拓者角色，实现了多项“首次”里程碑：

- **1957年**：发射世界上第一颗人造卫星 **斯普特尼克1号**，正式叩开太空时代的大门。
- **1961年**：尤里·加加林搭乘 **东方1号** 成为首位进入太空的人类，实现了载人航天的历史性突破。
- **后续成就**：诞生了首位女航天员（捷列什科娃，1963年）；发射了首个空间站 **礼炮1号**（1971年）及长期运行的 **和平号** 空间站（1986-2001年）；在深空探测领域，首次实现月球软着陆（月球9号）和首次金星着陆（金星7号）。

#### 二、 美国：登月壮举与深空领先（追赶与超越）

美国在初期追赶后，凭借强大的工业与科技实力，在人类登月和深空探测方面取得了领先优势：

- **1969年**：**阿波罗11号** 使命中，尼尔·阿姆斯特朗和巴兹·奥尔德林成为首次踏上月球的人类，书写了太空探索最辉煌的篇章。
- **深空探索与轨道设施**：发射 **旅行者号** 探测器飞往太阳系边缘；发展可重复使用的 **航天飞机**（1981-2011年），用于部署哈勃望远镜和建设**国际空间站**。
- **当代成就**：持续的火星探测（勇气号、好奇号、毅力号等系列火星车）；**新视野号** 飞越冥王星；以及重返月球的 **阿尔忒弥斯计划**。

#### 三、 欧洲：科学与工程并重的合作典范

欧洲太空局（ESA，1975年成立）走出了一条以科学探测和可靠火箭技术见长的道路：

- **代表性运载工具**：成功开发 